# Readme

# imports

In [ ]:
import photoproductionmodel as pm
import branching_ratios as br
import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mpl_toolkits import mplot3d
from google.colab import output
import time
from google.colab import files

output.enable_custom_widget_manager()


mp = 0.9382720813
e = 0.303
# gauge coupling charges
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}


#photoproduction model params CONVERGED
params = np.array([11.81336049,
                   3.92112205,
                   0.77877409,
                   0.68472406,
                   0.89693757,
                   0.80841249,
                   0.61509084,
                   0.69624785,
                   -4.21703614,
                   1.90289369,
                   8.81722149,
                   3.12168375,
                   1.1120275,
                   9.47975335,
                   0.92746299,
                   0.80066329,
                   0.80370305,
                   0.56739554,
                   1.74087675,
                   2.21138913,
                   5.52013592,
                   1.06419093,
                   0.99383077])

## Rotate 3d plots install (optional)

In [ ]:
# !pip install ipympl
'''
1. run the install above
2. restart kernel
3. do not run the install again
4. run %matplotlib widget
5. continue
'''

In [ ]:
# %matplotlib widget

# Acceptance function

In [ ]:
def A(cosX,EX,L0,mX,x,c_tau,l0=None):
  '''
  returns
    A(cosX,EX) the acceptance fraction or probability of decaying after shield and (if specified) before detector
  L0
    float distance from production point to end of shield
  c_tau
    float proper decay length
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  '''

  if cosX < 0:
    return np.nan

  gamma = EX/mX

  if not l0:
    return np.exp(-L0/(gamma*c_tau*cosX))

  elif l0:
    return np.exp(-L0/(gamma*c_tau*cosX)) - np.exp(-l0/(gamma*c_tau*cosX))

* removed alphaX dependency, now it directly takes c_tau
* maybe return 0 instead of nan for negative cosX if it helps with stuff down the pipeline

# Integration & Plotting

### submethods

In [ ]:
def kinematically_allowed_region_EX_cosX(s_tot,mX,resolution):
  '''
  s_tot
   float experimental c.o.m. energy squared
  mX
   float mass of X boson
  resolution
   int resolution of rectangular point cloud
  return
   EX_allowed, cosX_allowed numpy arrays of allowed coordinates in (EX,cosX)
  '''
  E_beam = (s_tot - mp**2) / (2 * mp)

  # uniform point cloud in rectangular region
  cosX = np.linspace(-1,1,resolution)
  EX = np.linspace(mX + 1e-6,E_beam,resolution)

  EX_grid, cosX_grid = np.meshgrid(EX,cosX)

  # point cloud limited to kinematically allowed region (1D objs: EX_allowed, cosX_allowed)
  qX = np.sqrt(EX_grid**2 - mX**2)
  nu = (mp * EX_grid - 0.5 * mX**2) / (mp - EX_grid + qX * cosX_grid)

  validity_mask = (nu>EX) & (nu<E_beam)
  EX_allowed, cosX_allowed = EX_grid[validity_mask] , cosX_grid[validity_mask]

  # redoing the above steps for strictly defined resolution in boundary given above
  # I want cosX spacing around 0.002, and EX spacing around 0.04, for all s_tot and mX
  target_dcos = 0.002
  target_dEX = 0.04

  cosX_allowed_min = np.min(cosX_allowed)
  cosX_allowed_max = np.max(cosX_allowed)
  N_cosX = int(np.ceil((cosX_allowed_max-cosX_allowed_min)/target_dcos)) + 1

  EX_allowed_min = np.min(EX_allowed)
  EX_allowed_max = np.max(EX_allowed)
  N_EX = int(np.ceil((EX_allowed_max-EX_allowed_min)/target_dEX)) + 1

  cosX2 = np.linspace(cosX_allowed_min,cosX_allowed_max,N_cosX)
  EX2 = np.linspace(EX_allowed_min,EX_allowed_max,N_EX)

  EX_grid2, cosX_grid2 = np.meshgrid(EX2,cosX2)

  qX2 = np.sqrt(EX_grid2**2 - mX**2)
  nu2 = (mp * EX_grid2 - 0.5 * mX**2) / (mp - EX_grid2 + qX2 * cosX_grid2)

  validity_mask2 = (nu2>EX2) & (nu2<E_beam)
  EX_allowed2, cosX_allowed2 = EX_grid2[validity_mask2] , cosX_grid2[validity_mask2]

  return EX_allowed2, cosX_allowed2

In [ ]:
def simple_kinematically_allowed_region_EX_cosX(s_tot,mX,resolution):
  '''
  No adaptive refinement, used for when I just need the boundaries
  s_tot
   float experimental c.o.m. energy squared
  mX
   float mass of X boson
  resolution
   int resolution of rectangular point cloud
  return
   EX_allowed, cosX_allowed numpy arrays of allowed coordinates in (EX,cosX)
  '''
  E_beam = (s_tot - mp**2) / (2 * mp)

  # uniform point cloud in rectangular region
  cosX = np.linspace(-1,1,resolution)
  EX = np.linspace(mX + 1e-6,E_beam,resolution)

  EX_grid, cosX_grid = np.meshgrid(EX,cosX)

  # point cloud limited to kinematically allowed region (1D objs: EX_allowed, cosX_allowed)
  qX = np.sqrt(EX_grid**2 - mX**2)
  nu = (mp * EX_grid - 0.5 * mX**2) / (mp - EX_grid + qX * cosX_grid)

  validity_mask = (nu>EX) & (nu<E_beam)
  EX_allowed, cosX_allowed = EX_grid[validity_mask] , cosX_grid[validity_mask]

  return EX_allowed, cosX_allowed

### plotting regions and integrands

In [ ]:
def plot_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq)

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, dsig,s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('dsigX_dEX_dcosX')
  plt.show()

In [ ]:
def plot_A(L0,l0,alphaX,mX,x):
  '''
  plots 3d point cloud over allowed region of (EX,cosX) of A(EX,cosX)

  L0
    float shield length
  alphaX
    float coupling constant
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  A_vals = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    A_vals[i] = A(cosX_allowed[i],EX_allowed[i],L0,l0,alphaX,mX,x)

  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, A_vals,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('A')
  point_cloud_ax.set_title('A')
  plt.show()

In [ ]:
def plot_A_times_dsigX_dEX_dcosX(s_tot,params,mX,xq,x,alphaX,L0,l0,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX * A(EX,cosX)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  A_times_dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    A_times_dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq) * A(cosX_allowed[i],EX_allowed[i],L0,l0,alphaX,mX,x)

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, A_times_dsig,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('A * dsigX_dEX_dcosX')
  plt.show()

In [ ]:
def plot_allowed_region(s_tot,mX,resolution):
  '''
  plots point cloud of allowed region
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  fig = plt.figure()
  plt.scatter(EX_allowed,cosX_allowed,s=0.1)
  plt.title('valid threshold')
  plt.xlabel('EX')
  plt.ylabel('cosX')

In [ ]:
def spline_region(s_tot,mX,resolution,plot = False):
  '''
  splines for dblquad integration
  plots splines of upper and lower EX(cosX)
  return sequence of CubicSpline objects (spline_EX_upper, spline_EX_lower)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  # interpolating region boundary functions of EX wrt cosX
  cosX_unique = np.unique(cosX_allowed)

  EX_upper = np.zeros(len(cosX_unique))
  EX_lower = np.zeros(len(cosX_unique)) # arrays of upper and lower values of EX for the set of unique cosXs in our point cloud

  for i in range(len(cosX_unique)):
    key = np.where(cosX_allowed == cosX_unique[i],True,False)
    EX_upper[i]= np.max(EX_allowed[key])
    EX_lower[i] = np.min(EX_allowed[key])

  spline_EX_upper = scipy.interpolate.CubicSpline(cosX_unique,EX_upper,extrapolate=False)
  spline_EX_lower = scipy.interpolate.CubicSpline(cosX_unique,EX_lower,extrapolate=False)

  if plot:
    fig = plt.figure()
    # plt.scatter(EX_upper,cosX_unique,s=0.1)
    cosX_range = np.linspace(np.min(cosX_unique),np.max(cosX_unique),resolution)
    plt.plot(spline_EX_upper(cosX_range),cosX_range,color='g')
    plt.plot(spline_EX_lower(cosX_range),cosX_range,color='r')
    plt.title('valid threshold, splines')
    plt.xlabel('EX')
    plt.ylabel('cosX')
    plt.show()

  return spline_EX_upper, spline_EX_lower

### computing integrals

In [ ]:
def dsig_integrand(EX,cosX,s_tot,params,mX,xq):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

In [ ]:
def dsig_times_acceptance_integrand(EX,cosX,s_tot,params,mX,xq,L0,l0,alphaX,x,c_tau):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq) * A(cosX,EX,L0,mX,x,c_tau)

In [ ]:
def integrate_dsig(s_tot,params,mX,xq):
  '''
  return
    float : mu_barns integrated cross section
  '''
  # region of integration
  cosX_lower_boundary = 0
  cosX_upper_boundary = 1
  EX_lower_boundary_func = mX + 1e-6
  EX_upper_boundary_func = (s_tot - mp**2) / (2 * mp)

  # integration
  result , error = scipy.integrate.dblquad(
    dsig_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq)
    )

  return result

In [ ]:
def rect_integrate_dsig_times_acceptance(s_tot,params,mX,xq,alphaX,x,L0,l0=None):

  '''
  integrates d_sigX_dEX_dcosX * A(EX,cosX) with scipy.dblquad

  L0
    float shield length
  alphaX
    float coupling constant
  mX
    float mass of X boson
  x
    ndarray of floats gauge couplings [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  return
    float value of integral
  '''

  # region of integration
  cosX_lower_boundary = 0
  cosX_upper_boundary = 1
  EX_lower_boundary_func = mX + 1e-6
  EX_upper_boundary_func = (s_tot - mp**2) / (2 * mp)

  # c tau independent of ex,cosx
  c_tau = br.get_decay_length(mX,alphaX,x) / 1e13 #cm
  print(f'c_tau in cm: {c_tau}, mass: {mX} , alphaX: {alphaX}')

  # integration
  result , error = scipy.integrate.dblquad(
    dsig_times_acceptance_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq,L0,l0,alphaX,x,c_tau)
    )
  return result


* updating dsig*acceptance integral to integrate over rectangular region

# Sampling (cosX,EX)

In [ ]:
def P(s_tot,EX,cosX,params,mX,xq):
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

In [ ]:
def mcmc_sample_EX_cosX(n,s_tot,params,mX,xq):
  '''
  return
    ndarray : nx2 posterior distribution (EX [GeV], cosX)
  n
    int : number of samples
  '''

  x_samples = np.zeros((n, 2))

  E_beam = (s_tot - mp**2) / (2 * mp)

  central_E = (E_beam - mX) / 2
  central_cos = 0.999
  standard_deviation_E = (E_beam-central_E)*0.67
  standard_deviation_cos = 0.01 #(1-central_cos)*0.67 # controversial? check?
  # print(f'err cosX is {standard_deviation_cos / np.sqrt(n)}') # errors
  # print(f'err EX is {standard_deviation_E / np.sqrt(n)}') # errors

  x_samples[0,0] = central_E # dummy first sample
  x_samples[0,1] = central_cos

  counter = 0
  for i in range(n - 1):
    EX_old = x_samples[i,0]
    cosX_old = x_samples[i,1]
    # print(EX_old, cosX_old) # diagnosis

    ## proposal ##
    EX_new = np.random.normal(central_E,standard_deviation_E)
    cosX_new = np.random.normal(central_cos,standard_deviation_cos)

    ## accept/reject for log ##
    A = P(s_tot,EX_new,cosX_new,params,mX,xq) / P(s_tot,EX_old,cosX_old,params,mX,xq)

    if A > 1:
      x_samples[i+1,0] = EX_new
      x_samples[i+1,1] = cosX_new
      counter +=1

    elif A > np.random.uniform(0,1):
      x_samples[i+1,0] = EX_new
      x_samples[i+1,1] = cosX_new
      counter +=1

    else:
      x_samples[i+1,0] = EX_old
      x_samples[i+1,1] = cosX_old
  # print('mcmc acceptance at mX = ' + str(mX) + ' is ' + str(counter/n)) # diagnosis

  return x_samples

In [ ]:
def estimate_expected_A(EX_cosX_samples,alphaX,mX,x,L0,l0=None):
  '''
  return
    float : cross section weighted average acceptance fraction
  '''

  EX_samples = EX_cosX_samples[:,0]
  cosX_samples = EX_cosX_samples[:,1]

  c_tau = br.get_decay_length(mX,alphaX,x) / 1e13 #fm to cm

  A_samples = np.zeros(len(EX_samples))
  for i in range(len(EX_samples)):
    A_samples[i] = A(cosX_samples[i],EX_samples[i],L0,mX,x,c_tau,l0)

  A_samples[np.isnan(A_samples)] = 0 # negative cosX values are returned as NaN by A().

  return np.sum(A_samples) / len(A_samples)

* statistical error is $\frac{σ}{\sqrt{n}}$
* refine these errors, and compute the error on crosssection-weighted-acceptance


##### analysis

* for the 2nd approach; E[A] must be estimated at every unique (mX,alphaX)
* sig_tot must be integrated once for every mX, samples must be taken for every mX
* for the 1st approach; sig_tot*A must be integrated once for every (mX,alphaX)

# Notes

##### improvements

* Integrating dsig times acceptance takes long, maybe replace this part with sampling.... see "results" section
* For integrating dsig, maybe just integrate over rectangular region: cosX(-1,1)EX(mX,EBeam). Instead of finding boundaries since pm.dsig() already returns zero outside boundaries

##### integration method

* Create a point cloud in a rectangular region in *(EX,cosX)* that supercedes the kinematically allowed region and remove all points that are not kinematically allowed.
* Create splines for the upper and lower curves of *EX with respect to cosX*

* Pass the integrand, the splines of *EX(cosX)*, and the upper and lower boundaries of *cosX* into ***scipy.integrate.dblquad(func, a, b, gfun, hfun)*** to perform the integral

##### results

$\frac{dN}{dE_XdcosX} = BR_{X→F} \cdot \ell \cdot \frac{d^2σ}{dE_XdcosX} \cdot A(E_X,cosX)$


---


$N = BR_{X→F} \cdot \ell \cdot \sigma \cdot E[A] $



---
$E[A] = \frac{\int∫A(E_X,cosX) \frac{d^2σ}{dE_XdcosX} dE_XdcosX}{\int\int\frac{d^2σ}{dE_XdcosX}dE_XdcosX} $

---
Two approaches
1. compute $N = BR_{X→F}\cdot\ell\cdot\int∫A(E_X,cosX) \frac{d^2σ}{dE_XdcosX} dE_XdcosX$
2. sample A to draw E[A], then compute $N = BR_{X→F}\cdot\ell\cdot E[A]\cdot\int\int\frac{d^2σ}{dE_XdcosX}dE_XdcosX$

* The second is faster incredibly more time efficient

##### plots

* For low masses the kinematically allowed region includes negative values of cosX. These are shown in the plots of the allowed region and of the double differential cross section, but are not shown in the plots including acceptance because it is 0 for negative cosX.
* When computing integrals for low masses where negative cosX is allowed, the boundary on cosX only includes the positive range

# Updates

* Fixed boundary resolution failure at low masses by implementing adaptive refinement.
* Fixed acceptance function definition, now defines the desired probability
* Fixed integration happening over negative cosX region, which occurs for low masses

# N heatmap

In [ ]:
# def N_1(int_luminosity,s_tot,params,mX,xq,L0,l0,alphaX,x,resolution):
#   '''
#   method 1
#   '''
#   BR = br.decay_profile(mX,alphaX,gauge_couplings['Chargephobic'])['BR_pi_plus_pi_minus']
#   sig_times_A_integral = integrate_dsig_times_acceptance(s_tot,params,mX,xq,L0,l0,alphaX,x,resolution)

#   return sig_times_A_integral * BR * int_luminosity * alphaX * 1e-30

In [ ]:
def N_2(kin_samples, int_sigma, BR, int_luminosity,mX,alphaX,x,L0,l0=None): # method 2
  '''
  int_luminosity
    float : cm^-2

  kin_samples
    ndarray : [EX GeV, cosX]

  int_sigma
    float : micro barns

  '''
  wav_A = estimate_expected_A(kin_samples,alphaX,mX,x,L0,l0)
  return int_luminosity * BR * int_sigma * wav_A

In [ ]:
def final_plot(int_luminosity,s_tot,params,xq,L0,l0,x):
  '''

  '''
  # axes
  divisions = 20
  mX_range = np.linspace(0 + 1e-6, 2, divisions)
  alphaX_range = np.logspace(-8, -1, divisions)


  # N
  N_grid = np.zeros((len(mX_range),len(alphaX_range)))

  for i in range(len(mX_range)):
    for j in range(len(alphaX_range)):
      N_grid[i,j] = N(int_luminosity,s_tot,params,
                      mX_range[i],
                      xq,L0,l0,
                      alphaX_range[j],
                      x,500)

  # plotting
  mX_grid,alphaX_grid = np.meshgrid(mX_range,alphaX_range,indexing='ij')

  print(N_grid)

  plt.pcolormesh(
    mX_grid,
    alphaX_grid,
    N_grid,
    shading='auto')

  plt.xlabel(r'$m_X$')
  plt.ylabel(r'$\alpha_X$')
  plt.colorbar(label='N')
  plt.yscale('log')
  plt.savefig('my_plot.png')
  plt.show()
  files.download('my_plot.png')

# final result

##walkthru
* step 1: for each mass, integrate cross section, sample kinematics, and determine branching ratio
* step 2: for each (mX,alphaX) pair, compute N : includes computing sigma weighted acceptance average
* step 3: plot it
---
##units
* N_2 expects int_luminosity in cm^-2, int_sigma in cm^2, L0 in cm
* Proper decay length (c_tau)-- *calculated in estimate_expected_A()*-- must be in cm
---
##experimental parameters
* planned Jefferson Laboratory 22 gev upgrade has reported luminosity to be $\mathcal{L} ≈ 10^{35} \rightarrow 10^{38}cm^{-2}s^{-1}$.
* we consider the high-luminosity operating scenario ($10^{38}cm^{-2}s^{-1}$), and a year long experiment ($10^7s$)
* integrated luminosity: $∫dt\mathcal{L}=\Delta t\cdot\mathcal{L} = 10^7s⋅10^{38}cm^{-2}s^{-1} = 10^{45}cm^-2$

In [ ]:
def plot_N_heatmap(int_luminosity,s_tot,params,x,xq,m_divs,alpha_divs,L0,l0=None):
  '''
  return
    None, plots heatmap
  int_luminosity
    float : cm^-2 integrated luminosity wrt to time duration of experiment
  s_tot
    float : GeV^2 squared center of mass energy of the experiment
  params
    ndarray : 20 parameters for photoproduction model (floats)
  x
    ndarray : gauge couplings to fermions [u,d,s,e,mu,nu_e,nu_mu,nu_tau]
  xq
    ndarray : gauge coulpings to quarks [u,d,s]
  L0
    float : centimeters minimum displaced production length (shield length)
  l0
    float : centimeters detecter and interaction seperation
  '''
  n = int(5e4) # number of samples (i benchmark 50k)


  alphaX_unique = np.logspace(-8,-1,alpha_divs)
  mX_unique = np.linspace(0+1e-6,2,m_divs)

  crosses = np.zeros(len(mX_unique)) # 'crosses' contains the integrated cross sections for each mass value in order of lowest mass to highest mass
  samples = np.zeros((len(mX_unique),n,2)) # 'samples' contains for each mass, an nx2 posterior distribution chain
  BR_unique = np.zeros(len(mX_unique)) # branching ratio to pi+pi-, using a dummy alphaX: it is not actually dependent on alphaX

  print(f'beginning mass dependent computations...')
  for i in range(0,len(mX_unique)):
    start_time = time.perf_counter() ###
    crosses[i] = integrate_dsig(s_tot,params,mX_unique[i],xq) * 1e-30 # microbarns -> cm^2
    samples[i] = mcmc_sample_EX_cosX(n,s_tot,params,mX_unique[i],xq)
    BR_unique[i] = br.decay_profile(mX_unique[i],1,x)['BR_pi_plus_pi_minus']
    end_time = time.perf_counter() ###
    print(f'{i}th mass complete, time elapsed: {end_time - start_time:.3}s') ###
  print(f'all masses complete') ###

  N_unique = np.zeros((len(alphaX_unique),len(mX_unique)))

  print(f'beginning (mX,alphaX) dependent computations...')
  start_time = time.perf_counter()
  for i in range(0,len(alphaX_unique)):
    for j in range(0,len(mX_unique)):
      N_unique[i,j] = N_2( #at this point, everything should be in correct units : int_lum in cm^-2 , crosses in cm^2 , L0 in cm
          samples[j],
          crosses[j],
          BR_unique[j],
          int_luminosity,
          mX_unique[j],
          alphaX_unique[i],
          x,
          L0,
          l0)
  end_time = time.perf_counter()
  print(f'(mX,alphaX) dependent computations complete, time elapsed: {end_time - start_time:.3}s')

  N_unique_masked = np.ma.masked_where(N_unique <= 0 , N_unique)
  plt.pcolormesh(mX_unique,alphaX_unique,N_unique_masked,shading='gouraud',norm=colors.LogNorm())
  plt.yscale('log')
  plt.colorbar(label = 'N')
  plt.xlabel(r'$m_X$')
  plt.ylabel(r'$\alpha_X$')
  plt.title(f'Luminosity: {int_luminosity*1e-7:.1e} cm$^{{-2}}$, Shield: {L0} cm, max N: {np.nanmax(N_unique)}')
  plt.savefig('my_plot.png')
  plt.show()
  files.download('my_plot.png')

int_luminosity = 1e45
s_tot = 42
x = gauge_couplings['Chargephobic']
xq = x[0:3]
m_divs = 20
alpha_divs = 20
L0 = 1 #cm

plot_N_heatmap(int_luminosity,s_tot,params,x,xq,m_divs,alpha_divs,L0)

## the diagnosis step is to find when / where / at what maasses the nans are happening. thx